## RQ-1: Validierung der automatisierten SEVQ-Evaluation

Dieses Notebook enthält alle Analysen zur ersten Forschungsfrage: Wie zuverlässig sind LLM-Judges als automatisierte Evaluatoren für generierte Datenvisualisierungen?

Die Grundidee ist, dass fünf verschiedene LLMs (GPT-5, GPT-4o, Gemini, Grok, Claude Sonnet) als "Judges" eingesetzt werden, die jede generierte Visualisierung nach dem SEVQ-Framework auf sechs Dimensionen bewerten. Um zu prüfen, ob diese automatisierten Bewertungen valide sind, werden sie mit den Bewertungen menschlicher Studienteilnehmer verglichen.

### Übersicht der Analysen

1. **ICC(2,1)** – Inter-Rater-Agreement der LLM-Judges, global und pro Konfiguration
2. **Boxplots** – Verteilung der Scores pro Konfiguration, Judge und Dimension
3. **Kriteriumsvalidität** – Vergleich der Judge-Mittelwerte mit den Studien-Mittelwerten
4. **Ensemble-Strategien** – Aggregationsansätze für einen robusteren Score
5. **Ergänzende Metriken** – Krippendorff Alpha, Spearman-Korrelation, Bland-Altman

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

---
### 0. Hilfsfunktionen

Zunächst werden alle Hilfsfunktionen definiert, die im weiteren Verlauf benötigt werden. Da gängige Bibliotheken für ICC und Krippendorff's Alpha Kompatibilitätsprobleme aufweisen (vor allem hinsichtlich Python-Versionsabhängigkeiten und unterschiedlicher Funktionssignaturen), wurden die Berechnungen auf Basis von `pingouin` und `krippendorff` neu implementiert und entsprechend dokumentiert.

Die Implementierungen unten kapseln jeweils einen Wrapper, der sicherstellt, dass die Bibliotheken mit unterschiedlichen Versionsständen umgehen kann. Das ist nötig, weil `krippendorff` in verschiedenen Versionen unterschiedliche Parameter-Namen für die Skalenebene verwendet.

In [ ]:
import numpy as np
import pandas as pd
import pingouin as pg
import krippendorff


def icc_2_1(ratings_matrix: np.ndarray):
    """
    Berechnet ICC(2,1) — Two-way random, single measures
    mit pingouin, aber behält die Eingabeform (n_targets × k_raters) bei.
    """

    n_targets, k_raters = ratings_matrix.shape

    # Matrix in Long-Format für pingouin umwandeln (intern)
    df_long = (
        pd.DataFrame(ratings_matrix)
        .reset_index(names="target")
        .melt(id_vars="target", var_name="rater", value_name="rating")
    )

    icc_df = pg.intraclass_corr(
        data=df_long,
        targets="target",
        raters="rater",
        ratings="rating"
    )

    res = icc_df.loc[icc_df["Type"] == "ICC2"]

    return {
        "icc": round(res["ICC"].iloc[0], 4),
        "F": round(res["F"].iloc[0], 4),
        "p": round(res["pval"].iloc[0], 4),
        "ci95_lower": round(res["CI95%"].iloc[0][0], 2),
        "ci95_upper": round(res["CI95%"].iloc[0][1], 2),
        "n_targets": int(n_targets),
        "k_raters": int(k_raters),
    }


def krippendorff_alpha(ratings_matrix, level="interval"):
    """Robuster Wrapper für verschiedene krippendorff-Versionen.
       Eingabe bleibt (n_targets x k_raters)."""
    data = ratings_matrix.T  # Bibliothek erwartet (n_raters × n_units)
    try:
        return round(krippendorff.alpha(reliability_data=data, level=level), 4)
    except TypeError:
        try:
            return round(krippendorff.alpha(reliability_data=data, metric=level), 4)
        except TypeError:
            return round(krippendorff.alpha(reliability_data=data), 4)

---
### Wichtige Konstanten

Hier werden die zentralen Konstanten für das gesamte Notebook festgelegt: die SEVQ-Dimensionen, das Mapping von Modell-IDs auf lesbare Namen sowie der Ausgabepfad für die generierten Abbildungen.

In [ ]:
METRICS = ['bugs', 'transformation', 'compliance', 'type', 'encoding', 'aesthetics']
MODEL_MAPPING = {
    'gpt-5': 'GPT-5',
    'gpt-4o': 'GPT-4o',
    'google/gemini-3-flash-preview': 'Gemini',
    'x-ai/grok-4.1-fast': 'Grok',
    'anthropic/claude-sonnet-4.5': 'Claude Sonnet'
}

OUTPUT_DIR = Path("./results/rq_1/figures")
OUTPUT_DIR.mkdir(exist_ok=True)

---
### 1. Daten laden

Im Folgenden werden die SEVQ-Bewertungen aus den Experimenten und die manuellen Bewertungen aus der Nutzerstudie eingelesen.

Zunächst werden die Pfade zu den vier Konfigurationsordnern definiert. Die Konfigurationen unterscheiden sich nach Sprache (Deutsch/Englisch) und Zielsprache für den generierten Code (R/Python).

In [ ]:
BASE_PATH = Path("./src/resources/paper_results/50_dataset_run")
STUDY_PATH = Path("./results/rq_1/study_data.csv")

CONFIGS = {
    'DE_R': BASE_PATH / 'DE' / 'R' / 'output',
    'DE_Python': BASE_PATH / 'DE' / 'Python' / 'output',
    'EN_R': BASE_PATH / 'EN' / 'R' / 'output',
    'EN_Python': BASE_PATH / 'EN' / 'Python' / 'output',
}

CONFIG_LABELS = {
    'DE_R': 'Deutsch & R',
    'DE_Python': 'Deutsch & Python',
    'EN_R': 'Englisch & R',
    'EN_Python': 'Englisch & Python',
}

In [ ]:
def load_sevq_data(output_path):
    """Lädt alle sevq.csv aus einem Konfigurationsordner."""
    dfs = []
    for folder in sorted(output_path.iterdir()):
        if not folder.is_dir():
            continue
        sevq_file = folder / 'sevq.csv'
        if sevq_file.exists():
            try:
                df = pd.read_csv(sevq_file)
                df['dataset'] = folder.name
                dfs.append(df)
            except Exception:
                pass
    if not dfs:
        return pd.DataFrame()
    result = pd.concat(dfs, ignore_index=True)
    result['model'] = result['model'].replace(MODEL_MAPPING)
    result['total'] = result[METRICS].sum(axis=1)
    return result

Jetzt werden alle SEVQ-Bewertungsdateien aus den Experimenten geladen. Für jede der vier Konfigurationen wird ausgegeben, wie viele Einzelbewertungen und wie viele einzigartige Visualisierungen vorhanden sind.

In [ ]:
all_sevq = {}
for config_key, config_path in CONFIGS.items():
    df = load_sevq_data(config_path)
    df['config'] = config_key
    all_sevq[config_key] = df
    print(f"  {CONFIG_LABELS[config_key]}: {len(df)} Bewertungen, "
          f"{df[['dataset','vis_index']].drop_duplicates().shape[0]} Visualisierungen")
    
# Combined DataFrame
df_all = pd.concat(all_sevq.values(), ignore_index=True)
df_all['vis_id'] = df_all['config'] + '_' + df_all['dataset'].astype(str) + '_' + df_all['vis_index'].astype(str)

print(f"\nGesamt: {len(df_all)} Bewertungen, "
      f"{df_all['vis_id'].nunique()} unique Visualisierungen, "
      f"{df_all['model'].nunique()} Judges")

In der Nutzerstudie wurden 12 der insgesamt 264 Visualisierungen von 14 menschlichen Teilnehmern bewertet. Diese Bewertungen dienen als Referenz für die Kriteriumsvalidität.

Die Studie-Daten liegen im Wide-Format vor und werden in ein Long-Format transformiert, das die weitere Analyse erleichtert.

In [ ]:
STUDY_MAPPING = [
    # (frage_nr, config, dataset, vis_index)
    (1,  'DE_R',      '139606', 0),
    (2,  'DE_R',      '104336', 0),
    (3,  'DE_R',      '103779', 1),
    (4,  'DE_Python', '16317',  0),
    (5,  'DE_Python', '108398', 1),
    (6,  'DE_Python', '103779', 2),
    (7,  'EN_R',      '137336', 0),
    (8,  'EN_R',      '104336', 0),
    (9,  'EN_R',      '103779', 1),
    (10, 'EN_Python', '13979',  0),
    (11, 'EN_Python', '104336', 0),
    (12, 'EN_Python', '103779', 1),
]


# Load study data
df_study_raw = pd.read_csv(STUDY_PATH, sep=';')
print(f"\nStudiendaten: {len(df_study_raw)} Teilnehmer, {df_study_raw.shape[1]} Spalten")

# Parse study data into long format
study_records = []
for frage_nr in range(1, 13):
    for _, row in df_study_raw.iterrows():
        participant_id = row['ID']
        scores = {}
        for dim in METRICS:
            dim_cap = dim.capitalize() if dim != 'type' else 'Type'
            col_name = f'Punkte für Bewerten Sie die Visualisierung - {dim_cap} / (Spalten 1-10)'
            # Find the right column for this question number
            # Columns repeat for each question, so we need to find the right occurrence
            matching_cols = [c for c in df_study_raw.columns if dim_cap in c and 'Punkte' in c]
            if len(matching_cols) >= frage_nr:
                col = matching_cols[frage_nr - 1]
                val = row[col]
                if pd.notna(val):
                    scores[dim] = int(val)

        if scores:
            scores['participant_id'] = participant_id
            scores['frage_nr'] = frage_nr
            study_records.append(scores)

df_study = pd.DataFrame(study_records)
df_study['total'] = df_study[METRICS].sum(axis=1)
print(f"Studienbewertungen (long format): {len(df_study)} Einträge")

---
### 2. Analyse 1: Inter-Rater-Agreement (ICC)

Wie einig sind sich die fünf LLM-Judges? Als Maß für das Inter-Rater-Agreement wird der ICC(2,1) (Two-Way Random, Single Measures) berechnet. Ergänzend wird Krippendorff's Alpha berechnet, das robuster gegenüber Skalenunterschieden ist.

**Global:** Zuerst wird das Agreement über alle vier Konfigurationen hinweg berechnet – also über alle 264 Visualisierungen.

In [ ]:
# Build ratings matrix: rows = visualizations, cols = judges
models = sorted(df_all['model'].unique())
vis_ids = sorted(df_all['vis_id'].unique())

# Matrix: avg total per (vis_id, model)
pivot_global = df_all.pivot_table(
    values='total', index='vis_id', columns='model', aggfunc='mean'
).dropna()

print(f"Matrix: {pivot_global.shape[0]} Visualisierungen x {pivot_global.shape[1]} Judges")

icc_global = icc_2_1(pivot_global.values)
print(f"ICC(2,1) global: {icc_global['icc']:.4f}")
print(f"  F = {icc_global['F']:.2f}, p = {icc_global['p']:.4f}")
print(f"  95% CI: [{icc_global['ci95_lower']:.2f}, {icc_global['ci95_upper']:.2f}]")
print(f"  n = {icc_global['n_targets']}, k = {icc_global['k_raters']}")

# Krippendorff Alpha global
ka_matrix = pivot_global.values.T
ka_global = krippendorff_alpha(ka_matrix)
print(f"Krippendorff's Alpha (global): {ka_global:.4f}")

**Interpretation:**
- Der ICC(2,1) von ~0.12 ist statistisch signifikant, aber inhaltlich sehr gering – die Judges sind sich kaum einig, was absolute Bewertungsniveaus betrifft.
- Krippendorff's Alpha liegt bei ~0.59 und verfehlt die übliche Akzeptanzschwelle von 0.667 knapp.
- Die Diskrepanz zwischen ICC und Alpha ist aufschlussreich: Die Judges operieren auf unterschiedlichen Skalenniveaus (Gemini bewertet generell höher als GPT-5), bewahren dabei aber eine ähnliche Rangfolge – Alpha ist weniger sensitiv gegenüber solchen Niveau-Unterschieden.

**Konfigurationsspezifisch:** Das gleiche Verfahren wird jetzt pro Konfiguration wiederholt, um zu sehen, ob bestimmte Konfigurationen zu mehr Übereinstimmung führen.

In [ ]:
icc_per_config = {}
for config_key, config_label in CONFIG_LABELS.items():
    df_cfg = df_all[df_all['config'] == config_key]
    pivot_cfg = df_cfg.pivot_table(
        values='total', index='vis_id', columns='model', aggfunc='mean'
    ).dropna()

    if pivot_cfg.shape[0] < 3:
        print(f"  {config_label}: zu wenig Daten ({pivot_cfg.shape[0]} Vis)")
        continue

    icc_result = icc_2_1(pivot_cfg.values)
    ka_result = krippendorff_alpha(pivot_cfg.values.T)
    icc_per_config[config_key] = icc_result

    print(f"  {config_label}: ICC(2,1) = {icc_result['icc']:.4f} "
          f"[{icc_result['ci95_lower']:.2f}, {icc_result['ci95_upper']:.2f}], "
          f"α_K = {ka_result:.4f}, "
          f"n = {icc_result['n_targets']}")

**Zusammenfassung per Konfiguration:**

| Konfiguration | n | ICC(2,1) [95% CI] | α_K | Einschätzung |
|---|---|---|---|---|
| Deutsch & R | 63 | 0.079 [0.01–0.18] | 0.675 | Sehr geringe ICC, Alpha knapp akzeptabel |
| Deutsch & Python | 69 | 0.137 [0.02–0.29] | 0.630 | Geringe Übereinstimmung, mäßige Reliabilität |
| Englisch & R | 69 | 0.132 [0.02–0.29] | 0.676 | Geringe ICC, moderat konsistent |
| Englisch & Python | 63 | 0.063 [0.00–0.15] | 0.723 | Niedrigste ICC, bestes Alpha |

Die Inter-Rater-Reliabilität ist in allen Konfigurationen gering bis moderat. Englisch & Python erzielt das beste Krippendorff-Alpha.

Zur Vollständigkeit wird das Agreement noch einmal nur für die 12 Visualisierungen aus der Studie berechnet – sowohl für die LLM-Judges als auch für die menschlichen Studienteilnehmer im direkten Vergleich.

In [ ]:
print("\n--- Nur Studien-Visualisierungen (n=12) ---")
study_vis_ids = []
for frage_nr, config, dataset, vis_idx in STUDY_MAPPING:
    vid = f"{config}_{dataset}_{vis_idx}"
    study_vis_ids.append(vid)

df_study_judges = df_all[df_all['vis_id'].isin(study_vis_ids)]
pivot_study = df_study_judges.pivot_table(
    values='total', index='vis_id', columns='model', aggfunc='mean'
).dropna()

icc_study_judges = icc_2_1(pivot_study.values)
ka_study = krippendorff_alpha(pivot_study.values.T)
print(f"ICC(2,1) Studien-Subset: {icc_study_judges['icc']:.4f} "
      f"[{icc_study_judges['ci95_lower']:.2f}, {icc_study_judges['ci95_upper']:.2f}]")
print(f"Krippendorff's Alpha: {ka_study:.4f}")

---
### Boxplots: Score-Verteilung pro Dimension und Judge

Um die Bewertungsunterschiede zwischen den Judges zu visualisieren, werden Boxplots pro Konfiguration, SEVQ-Dimension und Judge-Modell erstellt. Damit lassen sich systematische Bias-Muster erkennen.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(18, 14))
fig.suptitle('SEVQ-Bewertungen pro Dimension und Judge-Modell', fontsize=16, fontweight='bold')

for idx, (config_key, config_label) in enumerate(CONFIG_LABELS.items()):
    ax = axes[idx // 2][idx % 2]
    df_cfg = all_sevq[config_key]

    # Melt to long format
    df_long = df_cfg.melt(
        id_vars=['model', 'dataset', 'vis_index'],
        value_vars=METRICS,
        var_name='dimension',
        value_name='score'
    )

    sns.boxplot(
        data=df_long, x='dimension', y='score', hue='model',
        ax=ax, palette='Set2', linewidth=0.8
    )
    ax.set_title(config_label, fontsize=13, fontweight='bold')
    ax.set_xlabel('Dimension', fontsize=11)
    ax.set_ylabel('Score (1-10)', fontsize=11)
    ax.set_ylim(0, 11)
    ax.legend(fontsize=7, title='Judge', title_fontsize=8, loc='lower left')
    ax.tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'boxplots_dimension_judge.svg', bbox_inches='tight')
plt.close()
print(f"  Gespeichert: {OUTPUT_DIR / 'boxplots_dimension_judge.svg'}")

### Bias-Analyse

Wie stark weichen einzelne Judges im Durchschnitt voneinander ab? Die folgende Tabelle zeigt die mittleren Bewertungen pro Judge und Dimension. Ein zusätzlicher Boxplot visualisiert die Gesamtbewertung pro Judge über alle Konfigurationen.

In [ ]:
bias_table = df_all.groupby('model')[METRICS + ['total']].mean().round(2)
print(bias_table.to_string())

# Boxplot: Total-Score pro Judge über alle Konfigurationen
fig, ax = plt.subplots(figsize=(10, 6))
sns.boxplot(data=df_all, x='model', y='total', palette='Set2', ax=ax)
ax.set_title('Gesamtbewertung (Total) pro Judge-Modell über alle Konfigurationen',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Judge-Modell', fontsize=11)
ax.set_ylabel('Total Score (0-60)', fontsize=11)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'boxplot_total_per_judge.svg', bbox_inches='tight')
plt.close()
print(f"  Gespeichert: {OUTPUT_DIR / 'boxplot_total_per_judge.svg'}")

---
### 3. Analyse: Kriteriumsvalidität (LLM-Judges vs. Studie)

Bisher wurden die Judges untereinander verglichen. Jetzt stellt sich die entscheidende Frage: Stimmen die automatisierten LLM-Bewertungen mit den menschlichen Bewertungen aus der Studie überein?

Dazu wird für die 12 Visualisierungen, die in der Studie bewertet wurden, eine gemeinsame Vergleichstabelle erstellt.

In [ ]:
# gemeinsame vergleichstabelle
comparison_records = []

for frage_nr, config, dataset, vis_idx in STUDY_MAPPING:
    vid = f"{config}_{dataset}_{vis_idx}"

    # judge durchschnittliche Bewertung
    judge_data = df_all[
        (df_all['config'] == config) &
        (df_all['dataset'] == str(dataset)) &
        (df_all['vis_index'] == vis_idx)
    ]

    if judge_data.empty:
        print(f"  WARNUNG: Keine Judge-Daten für Frage {frage_nr} ({vid})")
        continue

    judge_avg_total = judge_data['total'].mean()
    judge_avg_dims = {dim: judge_data[dim].mean() for dim in METRICS}

    # Studie Teilnehmer Durchschnitt
    study_data_q = df_study[df_study['frage_nr'] == frage_nr]
    study_avg_total = study_data_q['total'].mean()
    study_avg_dims = {dim: study_data_q[dim].mean() for dim in METRICS}

    rec = {
        'frage': frage_nr,
        'config': CONFIG_LABELS[config],
        'dataset': dataset,
        'vis_index': vis_idx,
        'judge_total': round(judge_avg_total, 1),
        'study_total': round(study_avg_total, 1),
        'diff': round(judge_avg_total - study_avg_total, 1),
    }
    for dim in METRICS:
        rec[f'judge_{dim}'] = round(judge_avg_dims[dim], 1)
        rec[f'study_{dim}'] = round(study_avg_dims[dim], 1)

    comparison_records.append(rec)

df_comparison = pd.DataFrame(comparison_records)
print("\nVergleichstabelle (Ø Judge vs. Ø Studie):")
print(df_comparison[['frage', 'config', 'dataset', 'judge_total', 'study_total', 'diff']].to_string(index=False))

### Korrelation und Fehlermetriken

Wie stark korrelieren Judge- und Studienbewertungen? Neben Pearson- und Spearman-Korrelation werden MAE, RMSE und der mittlere Bias berechnet.

In [ ]:
from scipy import stats
import numpy as np

# Korrelationen (Robuste SciPy-Variante)
judge_totals = df_comparison['judge_total'].values
study_totals = df_comparison['study_total'].values

# Pearson- und Spearman-Korrelation
r_pearson, p_pearson = stats.pearsonr(judge_totals, study_totals)
r_spearman, p_spearman = stats.spearmanr(judge_totals, study_totals)

# Fehlermaße
mae = np.mean(np.abs(judge_totals - study_totals))
rmse = np.sqrt(np.mean((judge_totals - study_totals) ** 2))
mean_bias = np.mean(judge_totals - study_totals)

# Ausgabe
print(f"\nPearson r:  {r_pearson:.4f} (p = {p_pearson:.4f})")
print(f"Spearman ρ: {r_spearman:.4f} (p = {p_spearman:.4f})")
print(f"MAE:        {mae:.2f}")
print(f"RMSE:       {rmse:.2f}")
print(f"Mean Bias:  {mean_bias:.2f} (positiv = Judge bewertet höher)")

**Ergebnis:** Es besteht keine statistisch signifikante Korrelation zwischen den Judge- und Studienbewertungen. Der positive Bias zeigt, dass die Judges systematisch höher bewerten als die menschlichen Teilnehmer – im Schnitt um fast 6 Punkte.

Im Folgenden wird die Analyse auf Dimensionsebene aufgeschlüsselt, um zu prüfen, ob bestimmte SEVQ-Dimensionen besser übereinstimmen.

In [ ]:
for dim in METRICS:
    j_vals = df_comparison[f'judge_{dim}'].values
    s_vals = df_comparison[f'study_{dim}'].values
    r_s, p_s = stats.spearmanr(j_vals, s_vals)
    r_p, p_p = stats.pearsonr(j_vals, s_vals)
    print(f"  {dim:15s}: Pearson r = {r_p:.3f} (p={p_p:.3f}), Spearman ρ = {r_s:.3f} (p={p_s:.3f})")


**Ergebnis pro Dimension:**

| Dimension | Pearson r (p) | Spearman ρ (p) | Einschätzung |
|---|---|---|---|
| bugs | 0.074 (0.819) | -0.002 (0.996) | Keine Korrelation |
| transformation | 0.262 (0.411) | 0.321 (0.309) | Schwach positiv, nicht signifikant |
| compliance | -0.079 (0.807) | -0.244 (0.445) | Schwach negativ |
| type | -0.172 (0.594) | -0.273 (0.391) | Leicht negativ |
| encoding | -0.103 (0.750) | -0.124 (0.700) | Sehr schwach |
| aesthetics | -0.072 (0.824) | -0.168 (0.601) | Zufallsniveau |

In keiner Dimension besteht eine signifikante Übereinstimmung. Lediglich bei *Transformation* ist eine schwach positive Tendenz erkennbar.

### Bland-Altman-Plot

Da Korrelation allein keine Übereinstimmung misst, wird zusätzlich ein Bland-Altman-Plot erstellt. Dieser zeigt, ob die LLM-Bewertungen im Mittel systematisch von den menschlichen Urteilen abweichen und wie stark die Streuung der Differenzen ist.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Scatter Plot
ax1 = axes[0]
ax1.scatter(study_totals, judge_totals, s=80, c='#009B91', edgecolors='black', zorder=3)
min_val = min(min(study_totals), min(judge_totals)) - 3
max_val = max(max(study_totals), max(judge_totals)) + 3
ax1.plot([min_val, max_val], [min_val, max_val], 'k--', alpha=0.5, label='Perfekte Übereinstimmung')
ax1.set_xlabel('Ø Studienteilnehmer (Total)', fontsize=11)
ax1.set_ylabel('Ø LLM-Judges (Total)', fontsize=11)
ax1.set_title(f'Judge vs. Studie: Total Score\n'
              f'Pearson r = {r_pearson:.3f}, Spearman ρ = {r_spearman:.3f}',
              fontsize=12, fontweight='bold')
ax1.legend()
ax1.set_xlim(min_val, max_val)
ax1.set_ylim(min_val, max_val)
ax1.set_aspect('equal')
ax1.grid(True, alpha=0.3)

# Label points
for _, row in df_comparison.iterrows():
    ax1.annotate(f"F{int(row['frage'])}",
                 (row['study_total'], row['judge_total']),
                 textcoords="offset points", xytext=(5, 5), fontsize=8)

# Bland-Altman
ax2 = axes[1]
means = (judge_totals + study_totals) / 2
diffs = judge_totals - study_totals
mean_diff = np.mean(diffs)
std_diff = np.std(diffs, ddof=1)

ax2.scatter(means, diffs, s=80, c='#009B91', edgecolors='black', zorder=3)
ax2.axhline(mean_diff, color='#334152', linestyle='-', label=f'Mean Diff = {mean_diff:.1f}')
ax2.axhline(mean_diff + 1.96 * std_diff, color='gray', linestyle='--',
            label=f'+1.96 SD = {mean_diff + 1.96 * std_diff:.1f}')
ax2.axhline(mean_diff - 1.96 * std_diff, color='gray', linestyle='--',
            label=f'-1.96 SD = {mean_diff - 1.96 * std_diff:.1f}')
ax2.axhline(0, color='black', linestyle=':', alpha=0.3)
ax2.set_xlabel('Mittelwert (Judge + Studie) / 2', fontsize=11)
ax2.set_ylabel('Differenz (Judge − Studie)', fontsize=11)
ax2.set_title('Bland-Altman Plot\n(Systematische Abweichung & Grenzen der Übereinstimmung)',
              fontsize=12, fontweight='bold')
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)

for _, row in df_comparison.iterrows():
    m = (row['judge_total'] + row['study_total']) / 2
    d = row['judge_total'] - row['study_total']
    ax2.annotate(f"F{int(row['frage'])}", (m, d),
                 textcoords="offset points", xytext=(5, 5), fontsize=8)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'judge_vs_study.svg', bbox_inches='tight')
plt.close()
print(f"\n  Gespeichert: {OUTPUT_DIR / 'judge_vs_study.svg'}")

Abschließend wird geprüft, wie einig sich die Studienteilnehmer untereinander sind – zum Vergleich mit dem Judge-Agreement.

In [ ]:
study_pivot = df_study.pivot_table(
    values='total', index='frage_nr', columns='participant_id', aggfunc='mean'
).dropna(axis=1)

icc_study_participants = icc_2_1(study_pivot.values)
ka_study_part = krippendorff_alpha(study_pivot.values.T)
print(f"ICC(2,1) Studienteilnehmer: {icc_study_participants['icc']:.4f} "
      f"[{icc_study_participants['ci95_lower']:.2f}, {icc_study_participants['ci95_upper']:.2f}]")
print(f"Krippendorff's Alpha:        {ka_study_part:.4f}")

**Ergebnis:** Auch die Studienteilnehmer sind untereinander wenig einig (ICC < 0.37), was die Schwierigkeit der Aufgabe widerspiegelt. Das Judge-Agreement liegt in einem ähnlichen Bereich.

---
### 4. Ensemble-Strategien

Da kein einzelner Judge gleichzeitig eine hohe Rangkorrelation und einen geringen absoluten Fehler aufweist, werden verschiedene Ensemble-Strategien untersucht. Ziel ist es, durch Kombination mehrerer Judges eine Bewertung zu erhalten, die den menschlichen Urteilen möglichst nahekommt.

In [ ]:
# für alle 12 studien vis berechnen wir verschiedene aggregations der judges
ensemble_results = []

for frage_nr, config, dataset, vis_idx in STUDY_MAPPING:
    judge_data = df_all[
        (df_all['config'] == config) &
        (df_all['dataset'] == str(dataset)) &
        (df_all['vis_index'] == vis_idx)
    ]

    study_data_q = df_study[df_study['frage_nr'] == frage_nr]
    study_avg = study_data_q['total'].mean()

    totals_per_model = judge_data.groupby('model')['total'].mean()

    rec = {
        'frage': frage_nr,
        'study_avg': study_avg,
        'mean_all': totals_per_model.mean(),
        'median_all': totals_per_model.median(),
        'trimmed_mean': totals_per_model.sort_values().iloc[1:-1].mean() if len(totals_per_model) >= 3 else totals_per_model.mean(),
        'min_judge': totals_per_model.min(),
        'max_judge': totals_per_model.max(),
    }

    # Individual judges
    for model_name in models:
        if model_name in totals_per_model.index:
            rec[f'single_{model_name}'] = totals_per_model[model_name]

    ensemble_results.append(rec)

df_ensemble = pd.DataFrame(ensemble_results)

# Evaluate each strategy
strategies = ['mean_all', 'median_all', 'trimmed_mean'] + [f'single_{m}' for m in models if f'single_{m}' in df_ensemble.columns]
strategy_names = ['Mittelwert (alle)', 'Median (alle)', 'Getrimmtes Mittel'] + [f'Nur {m}' for m in models if f'single_{m}' in df_ensemble.columns]


In [ ]:
print(f"\n{'Strategie':<30s} {'Pearson r':>10s} {'Spearman ρ':>12s} {'MAE':>8s} {'RMSE':>8s} {'Bias':>8s}")
print("-" * 78)

best_mae = float('inf')
best_strategy = ''

for strat, name in zip(strategies, strategy_names):
    if strat not in df_ensemble.columns:
        continue
    vals = df_ensemble[strat].values
    study_vals = df_ensemble['study_avg'].values

    r_p, _ = stats.pearsonr(vals, study_vals)
    r_s, _ = stats.spearmanr(vals, study_vals)
    mae_s = np.mean(np.abs(vals - study_vals))
    rmse_s = np.sqrt(np.mean((vals - study_vals) ** 2))
    bias_s = np.mean(vals - study_vals)

    print(f"  {name:<28s} {r_p:>10.3f} {r_s:>12.3f} {mae_s:>8.2f} {rmse_s:>8.2f} {bias_s:>+8.2f}")

    if mae_s < best_mae:
        best_mae = mae_s
        best_strategy = name

print(f"\n  → Beste Strategie nach MAE: {best_strategy} (MAE = {best_mae:.2f})")

### Leave-One-Out Cross-Validation (LOO-CV)

Um die beste Ensemble-Gewichtung zu finden, ohne Überanpassung zu riskieren, wird eine LOO-CV durchgeführt: Für jede der 12 Visualisierungen wird auf Basis der verbleibenden 11 die optimale Gewichtung ermittelt und dann auf die ausgelassene Visualisierung angewendet.

In [ ]:
print("\n--- LOO-CV: Optimale Gewichtung ---")

loo_errors = {strat: [] for strat in strategies if strat in df_ensemble.columns}

for i in range(len(df_ensemble)):
    # Leave one out
    train = df_ensemble.drop(i)
    test = df_ensemble.iloc[i]

    for strat in loo_errors:
        pred = test[strat]
        actual = test['study_avg']
        loo_errors[strat].append(abs(pred - actual))

print(f"\n{'Strategie':<30s} {'LOO-MAE':>10s}")
print("-" * 42)
for strat, name in zip(strategies, strategy_names):
    if strat not in loo_errors:
        continue
    loo_mae = np.mean(loo_errors[strat])
    print(f"  {name:<28s} {loo_mae:>10.2f}")

---
## Visualisierungen für die Arbeit

Die folgenden Abbildungen werden für die Thesis generiert. Sie zeigen die wichtigsten Befunde kompakt und sind als SVG-Dateien gespeichert.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
heatmap_data = df_all.groupby('model')[METRICS].mean()
sns.heatmap(heatmap_data, annot=True, fmt='.1f', cmap='YlOrRd', ax=ax,
            vmin=5, vmax=10, linewidths=0.5)
ax.set_title('Durchschnittliche Bewertung pro Judge und Dimension (alle Konfigurationen)',
             fontsize=12, fontweight='bold')
ax.set_ylabel('Judge-Modell')
ax.set_xlabel('SEVQ-Dimension')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'heatmap_judge_dimension.svg', bbox_inches='tight')
plt.close()
print(f"  Gespeichert: {OUTPUT_DIR / 'heatmap_judge_dimension.svg'}")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Bland-Altman Plots pro SEVQ-Dimension (Judge vs. Studie)',
             fontsize=14, fontweight='bold')

for i, dim in enumerate(METRICS):
    ax = axes[i // 3][i % 3]

    j_vals = df_comparison[f'judge_{dim}'].values
    s_vals = df_comparison[f'study_{dim}'].values

    means_d = (j_vals + s_vals) / 2
    diffs_d = j_vals - s_vals
    mean_d = np.mean(diffs_d)
    std_d = np.std(diffs_d, ddof=1)

    ax.scatter(means_d, diffs_d, s=60, c='#009B91', edgecolors='black', zorder=3)
    ax.axhline(mean_d, color='#334152', linestyle='-', linewidth=1)
    ax.axhline(mean_d + 1.96 * std_d, color='gray', linestyle='--', linewidth=0.8)
    ax.axhline(mean_d - 1.96 * std_d, color='gray', linestyle='--', linewidth=0.8)
    ax.axhline(0, color='black', linestyle=':', alpha=0.3)

    r_s, _ = stats.spearmanr(j_vals, s_vals)
    ax.set_title(f'{dim.capitalize()}\nBias = {mean_d:.1f}, ρ = {r_s:.3f}', fontsize=11)
    ax.set_xlabel('Mittelwert', fontsize=9)
    ax.set_ylabel('Differenz (J−S)', fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'bland_altman_per_dimension.svg', bbox_inches='tight')
plt.close()
print(f"  Gespeichert: {OUTPUT_DIR / 'bland_altman_per_dimension.svg'}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
config_labels_list = [CONFIG_LABELS[k] for k in icc_per_config]
icc_values = [icc_per_config[k]['icc'] for k in icc_per_config]
ci_lower = [icc_per_config[k]['ci95_lower'] for k in icc_per_config]
ci_upper = [icc_per_config[k]['ci95_upper'] for k in icc_per_config]

yerr_lower = [max(0, v - l) for v, l in zip(icc_values, ci_lower)]
yerr_upper = [max(0, u - v) for v, u in zip(icc_values, ci_upper)]

bars = ax.bar(config_labels_list, icc_values, color='#009B91', edgecolor='black',
              yerr=[yerr_lower, yerr_upper], capsize=5)
ax.axhline(icc_global['icc'], color='#334152', linestyle='--',
           label=f'Global ICC = {icc_global["icc"]:.3f}')
ax.set_ylabel('ICC(2,1)', fontsize=11)
ax.set_title('Inter-Rater Agreement (ICC 2,1) der LLM-Judges pro Konfiguration',
             fontsize=12, fontweight='bold')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'icc_per_config.svg', bbox_inches='tight')
plt.close()
print(f"  Gespeichert: {OUTPUT_DIR / 'icc_per_config.svg'}")

---
## Zusammenfassung der bisherigen Ergebnisse

In [ ]:
print("\n" + "=" * 60)
print("ZUSAMMENFASSUNG")
print("=" * 60)
print(f"""
ICC(2,1) Global (alle Vis):          {icc_global['icc']:.4f}
Krippendorff's Alpha Global:         {ka_global:.4f}

ICC(2,1) Studien-Subset (n=12):      {icc_study_judges['icc']:.4f}
ICC(2,1) Studienteilnehmer (n=12):   {icc_study_participants['icc']:.4f}

Korrelation Judge vs. Studie:
  Pearson r:   {r_pearson:.4f} (p={p_pearson:.4f})
  Spearman ρ:  {r_spearman:.4f} (p={p_spearman:.4f})
  MAE:         {mae:.2f}
  Mean Bias:   {np.mean(judge_totals - study_totals):.2f}

Beste Ensemble-Strategie:            {best_strategy} (MAE = {best_mae:.2f})
""")

---
## Vertiefende Analysen

Im Folgenden werden die Ergebnisse weiter aufgeschlüsselt:
- Detaillierte Agreement-Analysen mit verschiedenen Toleranzschwellen
- Betrachtung auf Dimensionsebene
- Vergleich einzelner Judges mit dem Studien-Mittelwert
- Bin-basierte Übereinstimmung

### Agreement mit Toleranzbereich

Wie viele der 12 Visualisierungen werden "nah genug" an den menschlichen Bewertungen eingestuft? Dazu wird ein Toleranzparameter τ eingeführt: Zwei Bewertungen gelten als übereinstimmend, wenn ihre Differenz ≤ τ ist.

Zunächst wird die Gesamtbewertung (Total Score) betrachtet:

In [ ]:
judge_detail = []
for frage_nr, config, dataset, vis_idx in STUDY_MAPPING:
    jd = df_all[
        (df_all['config'] == config) &
        (df_all['dataset'] == str(dataset)) &
        (df_all['vis_index'] == vis_idx)
    ]
    for _, row in jd.iterrows():
        rec = {'frage': frage_nr, 'model': row['model'], 'total': row['total']}
        for dim in METRICS:
            rec[dim] = row[dim]
        judge_detail.append(rec)

df_judge_detail = pd.DataFrame(judge_detail)
study_avg_per_vis = df_study.groupby('frage_nr')[METRICS + ['total']].mean()
study_individual = df_study.copy()

comparison = []
for frage_nr, config, dataset, vis_idx in STUDY_MAPPING:
    jd = df_all[
        (df_all['config'] == config) &
        (df_all['dataset'] == str(dataset)) &
        (df_all['vis_index'] == vis_idx)
    ]
    judge_avg = jd['total'].mean()
    study_avg = study_avg_per_vis.loc[frage_nr, 'total']
    comparison.append({
        'frage': frage_nr,
        'config': config,
        'judge_total': judge_avg,
        'study_total': study_avg,
    })
    for dim in METRICS:
        comparison[-1][f'judge_{dim}'] = jd[dim].mean()
        comparison[-1][f'study_{dim}'] = study_avg_per_vis.loc[frage_nr, dim]

df_comp = pd.DataFrame(comparison)

In [ ]:
tolerances_total = [0, 1, 2, 3, 5, 7, 10]
print(f"{'τ (Toleranz)':>14s} {'Agreement':>12s} {'Anzahl':>10s}")
print("-" * 40)
for tau in tolerances_total:
    agree = np.sum(np.abs(df_comp['judge_total'] - df_comp['study_total']) <= tau)
    agreement = agree / len(df_comp)
    print(f"  τ = {tau:>5.0f}       {agreement:>10.3f}    {agree:>5d}/12")


**Ergebnis:** Erst bei einem Toleranzbereich von τ = 7 werden zwei Drittel der Bewertungen als übereinstimmend gewertet – die Judges müssen also eine beträchtliche Abweichung toleriert werden, bevor Agreement entsteht.

### Agreement auf Dimensionsebene

In [ ]:
print("\n\n--- Agreement auf Dimensionsebene (Ø Judge vs. Ø Studie pro Dimension) ---")
tolerances_dim = [0, 0.5, 1, 1.5, 2]

for dim in METRICS:
    j_vals = df_comp[f'judge_{dim}'].values
    s_vals = df_comp[f'study_{dim}'].values
    diffs = np.abs(j_vals - s_vals)

    print(f"\n  {dim.upper()}: Mean Diff = {np.mean(j_vals - s_vals):+.2f}, MAE = {np.mean(diffs):.2f}")
    for tau in tolerances_dim:
        agree = np.sum(diffs <= tau)
        print(f"    τ = {tau:.1f}: Agreement = {agree/len(diffs):.3f} ({agree}/12)")

**Fazit:** Auf Dimensionsebene zeigt sich ein differenzierteres Bild. Bei *Transformation* sind die mittleren Bewertungen nahezu identisch, bei *Type* dagegen größere Abweichungen. Für Bugs, Encoding und Aesthetics wird bei τ = 2 in der Mehrzahl der Fälle Agreement erreicht.

### Agreement pro einzelnem Judge

In [ ]:
print("\n\n--- Agreement pro einzelnem Judge (Total) ---")
print(f"{'Judge':<20s}", end="")
for tau in [3, 5, 7, 10]:
    print(f"  {'τ='+str(tau):>8s}", end="")
print(f"  {'MAE':>8s}  {'Bias':>8s}  {'Spearman':>10s}")
print("-" * 88)

judge_agreements = {}
for model in models:
    model_totals = []
    for frage_nr, config, dataset, vis_idx in STUDY_MAPPING:
        jd = df_all[
            (df_all['config'] == config) &
            (df_all['dataset'] == str(dataset)) &
            (df_all['vis_index'] == vis_idx) &
            (df_all['model'] == model)
        ]
        if len(jd) > 0:
            model_totals.append(jd['total'].values[0])
        else:
            model_totals.append(np.nan)

    model_totals = np.array(model_totals)
    study_totals = df_comp['study_total'].values
    diffs = model_totals - study_totals
    abs_diffs = np.abs(diffs)

    judge_agreements[model] = model_totals

    print(f"  {model:<18s}", end="")
    for tau in [3, 5, 7, 10]:
        agree = np.sum(abs_diffs <= tau) / len(abs_diffs)
        print(f"  {agree:>8.3f}", end="")

    mae = np.nanmean(abs_diffs)
    bias = np.nanmean(diffs)
    rho, _ = stats.spearmanr(model_totals[~np.isnan(model_totals)],
                            study_totals[~np.isnan(model_totals)])
    print(f"  {mae:>8.2f}  {bias:>+8.2f}  {rho:>10.3f}")

**Zusammenfassung:** GPT-4o und GPT-5 liegen tendenziell am nächsten an den menschlichen Urteilen (MAE ≈ 8), Claude liegt im mittleren Bereich, während Gemini und Grok deutlich höhere Abweichungen und einen stärkeren positiven Bias zeigen. Gemini besitzt jedoch die höchste Rangkorrelation.

### Bin-basiertes Agreement

Als Alternative zum numerischen Vergleich wird geprüft, ob die Bewertungen in kategoriale Bereiche ("Low / Medium / High") eingeteilt werden können und ob dann mehr Übereinstimmung entsteht.

In [ ]:
print("\n\n--- Agreement auf Binned Total Score ---")
print("  Bins: Low (0-20), Medium (21-40), High (41-60)")

def bin_score(score):
    if score <= 20:
        return 'Low'
    elif score <= 40:
        return 'Medium'
    else:
        return 'High'

j_binned = [bin_score(x) for x in df_comp['judge_total']]
s_binned = [bin_score(x) for x in df_comp['study_total']]
exact_agree_binned = sum(1 for a, b in zip(j_binned, s_binned) if a == b)
print(f"  Agreement (3 Bins): {exact_agree_binned/12:.3f} ({exact_agree_binned}/12)")

# Finer bins
print("\n  Bins: Very Low (0-15), Low (16-30), Medium (31-45), High (46-60)")
def bin_score_4(score):
    if score <= 15: return 'Very Low'
    elif score <= 30: return 'Low'
    elif score <= 45: return 'Medium'
    else: return 'High'

j_binned4 = [bin_score_4(x) for x in df_comp['judge_total']]
s_binned4 = [bin_score_4(x) for x in df_comp['study_total']]
exact_agree_4 = sum(1 for a, b in zip(j_binned4, s_binned4) if a == b)
print(f"  Agreement (4 Bins): {exact_agree_4/12:.3f} ({exact_agree_4}/12)")

In [ ]:
print("\n\n--- Exakte Agreement pro Dimension (gerundet auf Integer) ---")
print("  S_llm = round(Ø Judge), S_human = round(Ø Studie)")

for dim in METRICS:
    j_rounded = np.round(df_comp[f'judge_{dim}'].values).astype(int)
    s_rounded = np.round(df_comp[f'study_{dim}'].values).astype(int)
    exact = np.sum(j_rounded == s_rounded)
    within_1 = np.sum(np.abs(j_rounded - s_rounded) <= 1)
    print(f"  {dim:15s}: Exakt = {exact/12:.3f} ({exact}/12), "
          f"±1 = {within_1/12:.3f} ({within_1}/12)")

### Bias-Korrektur

Jeder Judge bewertet im Schnitt auf einem anderen Niveau (Gemini zu hoch, GPT-5 zu niedrig). Die Bias-Korrektur zieht von jedem Judge-Score dessen modellspezifische Abweichung zum globalen Gesamtmittel ab. Das normiert die Scores, ohne die Rangfolge zu verändern.

In [ ]:
print("  Für jeden Judge: Score_korrigiert = Score - (Ø_Judge_global - Ø_Gesamt)")

# Global average across all judges and all visualizations
global_mean_total = df_all['total'].mean()
judge_means = df_all.groupby('model')['total'].mean()

print(f"\n  Globaler Mittelwert: {global_mean_total:.1f}")
print("  Judge-Mittelwerte und Bias:")
for model in models:
    bias = judge_means[model] - global_mean_total
    print(f"    {model:<18s}: Ø = {judge_means[model]:.1f}, Bias = {bias:+.1f}")


# Apply bias correction and recalculate ensemble
bias_corrected_totals = {}
for model in models:
    bias = judge_means[model] - global_mean_total
    corrected = []
    for frage_nr, config, dataset, vis_idx in STUDY_MAPPING:
        jd = df_all[
            (df_all['config'] == config) &
            (df_all['dataset'] == str(dataset)) &
            (df_all['vis_index'] == vis_idx) &
            (df_all['model'] == model)
        ]
        if len(jd) > 0:
            corrected.append(jd['total'].values[0] - bias)
        else:
            corrected.append(np.nan)
    bias_corrected_totals[model] = np.array(corrected)

#print(bias_corrected_totals)

### Z-Score-Normalisierung

Eine weitergehende Normalisierung skaliert jeden Judge-Score zusätzlich nach der modellspezifischen Standardabweichung. So werden nicht nur Mittelwert-Unterschiede, sondern auch Unterschiede in der Spreizung der Bewertungen angeglichen.

In [ ]:
print("  Score_norm = (Score - Ø_Judge) / SD_Judge * SD_target + Ø_target")

judge_stds = df_all.groupby('model')['total'].std()
# Target: Study participant distribution
target_mean = df_comp['study_total'].mean()
target_std = df_comp['study_total'].std()
print(f"  Ziel-Verteilung (Studie): Ø = {target_mean:.1f}, SD = {target_std:.1f}")

zscore_totals = {}
for model in models:
    m = judge_means[model]
    s = judge_stds[model]
    normalized = []
    for frage_nr, config, dataset, vis_idx in STUDY_MAPPING:
        jd = df_all[
            (df_all['config'] == config) &
            (df_all['dataset'] == str(dataset)) &
            (df_all['vis_index'] == vis_idx) &
            (df_all['model'] == model)
        ]
        if len(jd) > 0 and s > 0:
            z = (jd['total'].values[0] - m) / s
            normalized.append(z * target_std + target_mean)
        else:
            normalized.append(np.nan)
    zscore_totals[model] = np.array(normalized)

### Lineare Kalibrierung (LOO)

Die stärkste Form der Anpassung: Für jeden Judge wird eine lineare Funktion geschätzt (OLS), die seine Scores auf den menschlichen Maßstab kalibriert. Um Überanpassung zu vermeiden, werden die Kalibrierungsparameter jeweils auf den verbleibenden n-1 Beobachtungen geschätzt (Leave-One-Out).

In [ ]:
def loo_linear_calibrate(judge_scores, human_scores):
    """LOO-CV: Für jede Vis, kalibriere auf den restlichen 11 und sage vorher."""
    n = len(judge_scores)
    predictions = np.zeros(n)

    for i in range(n):
        # Train on all except i
        mask = np.ones(n, dtype=bool)
        mask[i] = False
        x_train = judge_scores[mask]
        y_train = human_scores[mask]

        # Simple OLS: y = a*x + b
        x_mean = x_train.mean()
        y_mean = y_train.mean()
        ss_xx = np.sum((x_train - x_mean) ** 2)
        if ss_xx == 0:
            predictions[i] = y_mean
        else:
            a = np.sum((x_train - x_mean) * (y_train - y_mean)) / ss_xx
            b = y_mean - a * x_mean
            predictions[i] = a * judge_scores[i] + b

    return predictions

study_totals = df_comp['study_total'].values
calibrated_totals = {}
for model in models:
    j_scores = np.array(judge_agreements[model], dtype=float)
    valid = ~np.isnan(j_scores)
    if valid.sum() >= 4:
        preds = loo_linear_calibrate(j_scores[valid], study_totals[valid])
        calibrated_totals[model] = preds
    else:
        calibrated_totals[model] = j_scores

---
## Auswertung aller Strategien

Alle oben definierten Strategien werden systematisch ausgewertet und nach verschiedenen Metriken verglichen: Pearson- und Spearman-Korrelation, MAE, RMSE, Bias und Agreement bei verschiedenen Toleranzschwellen.

In [ ]:
strategies = {}

# --- Baselines ---
strategies['Ø Alle Judges (roh)'] = df_comp['judge_total'].values

# --- Einzelne Judges (roh) ---
for model in models:
    strategies[f'Nur {model} (roh)'] = np.array(judge_agreements[model])

# --- Bias-korrigiert: Mittelwert ---
bc_mean = np.nanmean([bias_corrected_totals[m] for m in models], axis=0)
strategies['Ø Bias-korrigiert'] = bc_mean

# --- Bias-korrigiert: Einzelne ---
for model in models:
    strategies[f'{model} (bias-korr.)'] = bias_corrected_totals[model]

# --- Z-Score-normalisiert: Mittelwert ---
zs_mean = np.nanmean([zscore_totals[m] for m in models], axis=0)
strategies['Ø Z-Score-norm.'] = zs_mean

# --- Z-Score: Einzelne ---
for model in models:
    strategies[f'{model} (z-norm.)'] = zscore_totals[model]

# --- Kalibriert (LOO): Einzelne ---
for model in models:
    strategies[f'{model} (kalibriert LOO)'] = calibrated_totals[model]

# --- Kalibriert: Mittelwert der kalibrierten Judges ---
cal_mean = np.nanmean([calibrated_totals[m] for m in models], axis=0)
strategies['Ø Kalibriert (LOO)'] = cal_mean

# --- Median ---
raw_per_model = np.array([judge_agreements[m] for m in models])
strategies['Median (roh)'] = np.nanmedian(raw_per_model, axis=0)

# --- Trimmed Mean (ohne höchsten und niedrigsten) ---
sorted_scores = np.sort(raw_per_model, axis=0)
if len(models) >= 3:
    strategies['Getrimmtes Mittel (roh)'] = np.mean(sorted_scores[1:-1], axis=0)

In [ ]:
from itertools import combinations

# --- Subset + Bias-Korrektur ---
for k in range(2, len(models) + 1):
    best_mae_k = float('inf')
    best_combo_k = None
    best_vals_k = None

    for combo in combinations(range(len(models)), k):
        combo_models = [models[i] for i in combo]
        combo_scores = np.array([bias_corrected_totals[m] for m in combo_models])
        combo_mean = np.nanmean(combo_scores, axis=0)
        mae_k = np.nanmean(np.abs(combo_mean - study_totals))

        if mae_k < best_mae_k:
            best_mae_k = mae_k
            best_combo_k = combo_models
            best_vals_k = combo_mean

    combo_name = '+'.join([m.split()[0] if ' ' not in m else m for m in best_combo_k])
    strategies[f'Bestes {k}er-Subset (bc): {combo_name}'] = best_vals_k

In [ ]:
# --- Subset-Selektion: Alle 2er, 3er, 4er Kombinationen ---
best_subset_results = {}
for k in range(2, len(models) + 1):
    best_mae_k = float('inf')
    best_combo_k = None
    best_vals_k = None

    for combo in combinations(range(len(models)), k):
        combo_models = [models[i] for i in combo]
        combo_scores = np.array([judge_agreements[m] for m in combo_models])
        combo_mean = np.nanmean(combo_scores, axis=0)
        mae_k = np.nanmean(np.abs(combo_mean - study_totals))

        if mae_k < best_mae_k:
            best_mae_k = mae_k
            best_combo_k = combo_models
            best_vals_k = combo_mean

    best_subset_results[k] = (best_combo_k, best_mae_k, best_vals_k)
    combo_name = '+'.join([m.split()[0] if ' ' not in m else m for m in best_combo_k])
    strategies[f'Bestes {k}er-Subset (roh): {combo_name}'] = best_vals_k

In [ ]:
# --- Subset + Z-Score ---
for k in range(2, len(models) + 1):
    best_mae_k = float('inf')
    best_combo_k = None
    best_vals_k = None

    for combo in combinations(range(len(models)), k):
        combo_models = [models[i] for i in combo]
        combo_scores = np.array([zscore_totals[m] for m in combo_models])
        combo_mean = np.nanmean(combo_scores, axis=0)
        mae_k = np.nanmean(np.abs(combo_mean - study_totals))

        if mae_k < best_mae_k:
            best_mae_k = mae_k
            best_combo_k = combo_models
            best_vals_k = combo_mean

    combo_name = '+'.join([m.split()[0] if ' ' not in m else m for m in best_combo_k])
    strategies[f'Bestes {k}er-Subset (zn): {combo_name}'] = best_vals_k

In [ ]:
# --- Majority Vote auf Bin-Ebene ---
# For each vis, each judge votes for a bin, then majority wins
def majority_vote_binned(judge_dict, bin_func, models_list):
    results = []
    for i in range(12):
        votes = []
        for m in models_list:
            if not np.isnan(judge_dict[m][i]):
                votes.append(bin_func(judge_dict[m][i]))
        if votes:
            # Most common vote
            from collections import Counter
            c = Counter(votes)
            winner = c.most_common(1)[0][0]
            # Map back to midpoint of bin
            midpoints = {'Low': 10, 'Medium': 30, 'High': 50,
                         'Very Low': 7.5, 'Low': 22.5, 'Medium': 37.5, 'High': 52.5}
            results.append(midpoints.get(winner, 37.5))
        else:
            results.append(np.nan)
    return np.array(results)

---
## Ergebnisse aller Strategien im Vergleich

In [ ]:
print(f"\n{'Strategie':<50s} {'Pearson':>8s} {'Spearman':>9s} {'MAE':>7s} {'RMSE':>7s} {'Bias':>7s} {'Agr τ=5':>8s} {'Agr τ=7':>8s}")
print("-" * 108)

eval_results = []

for name, vals in strategies.items():
    vals = np.asarray(vals, dtype=float)
    valid = ~np.isnan(vals)
    if valid.sum() < 4:
        continue

    v = vals[valid]
    s = study_totals[valid]

    r_p, p_p = stats.pearsonr(v, s)
    r_s, p_s = stats.spearmanr(v, s)
    mae = np.mean(np.abs(v - s))
    rmse = np.sqrt(np.mean((v - s) ** 2))
    bias = np.mean(v - s)
    agr5 = np.sum(np.abs(v - s) <= 5) / len(v)
    agr7 = np.sum(np.abs(v - s) <= 7) / len(v)

    eval_results.append({
        'name': name, 'pearson': r_p, 'spearman': r_s,
        'mae': mae, 'rmse': rmse, 'bias': bias,
        'agr5': agr5, 'agr7': agr7
    })

    print(f"  {name:<48s} {r_p:>8.3f} {r_s:>9.3f} {mae:>7.2f} {rmse:>7.2f} {bias:>+7.2f} {agr5:>8.3f} {agr7:>8.3f}")

df_eval = pd.DataFrame(eval_results)


In [ ]:
print("\n\n--- Top 10 nach MAE ---")
top_mae = df_eval.nsmallest(10, 'mae')
for i, row in top_mae.iterrows():
    print(f"  {row['name']:<50s} MAE={row['mae']:.2f}, ρ={row['spearman']:.3f}, Agr(τ=5)={row['agr5']:.3f}")

print("\n--- Top 10 nach Spearman ρ ---")
top_spearman = df_eval.nlargest(10, 'spearman')
for i, row in top_spearman.iterrows():
    print(f"  {row['name']:<50s} ρ={row['spearman']:.3f}, MAE={row['mae']:.2f}, Agr(τ=5)={row['agr5']:.3f}")

print("\n--- Top 10 nach Agreement (τ=5) ---")
top_agr = df_eval.nlargest(10, 'agr5')
for i, row in top_agr.iterrows():
    print(f"  {row['name']:<50s} Agr(τ=5)={row['agr5']:.3f}, MAE={row['mae']:.2f}, ρ={row['spearman']:.3f}")



### Visualisierungen: Strategievergleich

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
fig.suptitle('Vergleich der Ensemble-Strategien zur SEVQ-Validierung', fontsize=14, fontweight='bold')

# Group strategies by category
categories = {
    'Roh (unkorrigiert)': ['Ø Alle Judges (roh)', 'Median (roh)', 'Getrimmtes Mittel (roh)'],
    'Bias-korrigiert': ['Ø Bias-korrigiert'],
    'Z-Score-normalisiert': ['Ø Z-Score-norm.'],
    'Kalibriert (LOO-CV)': ['Ø Kalibriert (LOO)'],
}

# Add single judges
for model in models:
    categories['Roh (unkorrigiert)'].append(f'Nur {model} (roh)')

In [ ]:
# Bar chart: MAE by strategy category
cat_results = {}
for cat_name, strat_names in categories.items():
    maes = []
    for sn in strat_names:
        match = df_eval[df_eval['name'] == sn]
        if len(match) > 0:
            maes.append(match['mae'].values[0])
    if maes:
        cat_results[cat_name] = np.mean(maes)

ax = axes[0]
ax.barh(list(cat_results.keys()), list(cat_results.values()), color=['#009B91', 'coral', 'gold', 'mediumpurple'])
ax.set_xlabel('MAE (niedriger = besser)', fontsize=10)
ax.set_title('MAE pro Strategiekategorie (Ø)', fontsize=11, fontweight='bold')
ax.grid(axis='x', alpha=0.3)

In [ ]:
# Scatter: all strategies - MAE vs Spearman
ax = axes[1]
for _, row in df_eval.iterrows():
    color = '#009B91'
    if 'bias-korr' in row['name'] or 'Bias-korr' in row['name']:
        color = 'coral'
    elif 'z-norm' in row['name'] or 'Z-Score' in row['name']:
        color = 'gold'
    elif 'kalibriert' in row['name'] or 'Kalibriert' in row['name']:
        color = 'mediumpurple'
    ax.scatter(row['mae'], row['spearman'], c=color, s=50, edgecolors='black', linewidth=0.5, alpha=0.7)
ax.set_xlabel('MAE (niedriger = besser)', fontsize=10)
ax.set_ylabel('Spearman ρ (höher = besser)', fontsize=10)
ax.set_title('Alle Strategien: MAE vs. Spearman', fontsize=11, fontweight='bold')
ax.axhline(0, color='gray', linestyle=':', alpha=0.5)
ax.grid(True, alpha=0.3)

In [ ]:
# Legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#009B91', label='Roh'),
    Patch(facecolor='coral', label='Bias-korrigiert'),
    Patch(facecolor='gold', label='Z-Score'),
    Patch(facecolor='mediumpurple', label='Kalibriert'),
]
ax.legend(handles=legend_elements, fontsize=8)

# Agreement comparison for key strategies
ax = axes[2]
key_strats = ['Ø Alle Judges (roh)', 'Ø Bias-korrigiert', 'Ø Z-Score-norm.', 'Ø Kalibriert (LOO)']
key_labels = ['Roh', 'Bias-korr.', 'Z-Score', 'Kalibriert']
taus = [3, 5, 7, 10]

agreement_data = {}
for strat, label in zip(key_strats, key_labels):
    if strat in strategies:
        vals = strategies[strat]
        valid = ~np.isnan(vals)
        v, s = vals[valid], study_totals[valid]
        agreement_data[label] = [np.sum(np.abs(v - s) <= tau) / len(v) for tau in taus]

x = np.arange(len(taus))
width = 0.18
for i, (label, agr_vals) in enumerate(agreement_data.items()):
    ax.bar(x + i * width, agr_vals, width, label=label)

ax.set_xlabel('Toleranz τ', fontsize=10)
ax.set_ylabel('Agreement', fontsize=10)
ax.set_title('Agreement nach Toleranz und Strategie', fontsize=11, fontweight='bold')
ax.set_xticks(x + width * 1.5)
ax.set_xticklabels([f'τ={t}' for t in taus])
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3)
ax.set_ylim(0, 1.05)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'ensemble_strategies_comparison.svg', bbox_inches='tight')
plt.close()
print(f"\n\nGespeichert: {OUTPUT_DIR / 'ensemble_strategies_comparison.svg'}")

### Detailvergleich: Beste Strategie vs. Studie

Die beste Strategie nach MAE wird mit den Studienbewertungen direkt verglichen – pro Visualisierung und im Kontext mit dem rohen Judges-Mittelwert.

In [ ]:
# Find best strategy by MAE
best_row = df_eval.loc[df_eval['mae'].idxmin()]
best_name = best_row['name']
best_vals = strategies[best_name]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Bar chart: comparison per visualization
ax = axes[0]
x = np.arange(12)
width = 0.35
bars1 = ax.bar(x - width/2, study_totals, width, label='Studienteilnehmer (Ø)', color='#009B91', edgecolor='black')
bars2 = ax.bar(x + width/2, best_vals, width, label=f'{best_name}', color='coral', edgecolor='black')
ax.set_xlabel('Frage Nr.', fontsize=10)
ax.set_ylabel('Total Score (0-60)', fontsize=10)
ax.set_title(f'Beste Strategie vs. Studie\n({best_name})', fontsize=11, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels([f'F{i+1}' for i in range(12)])
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3)

# Also compare with raw mean
raw_vals = strategies['Ø Alle Judges (roh)']
ax = axes[1]
bars1 = ax.bar(x - width, study_totals, width, label='Studie', color='#009B91', edgecolor='black')
bars2 = ax.bar(x, raw_vals, width, label='Ø Judges (roh)', color='lightcoral', edgecolor='black')
bars3 = ax.bar(x + width, best_vals, width, label=best_name, color='coral', edgecolor='black')
ax.set_xlabel('Frage Nr.', fontsize=10)
ax.set_ylabel('Total Score (0-60)', fontsize=10)
ax.set_title('Vergleich: Studie vs. Roh vs. Beste Strategie', fontsize=11, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels([f'F{i+1}' for i in range(12)])
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'best_strategy_detail.svg', bbox_inches='tight')
plt.close()
print(f"Gespeichert: {OUTPUT_DIR / 'best_strategy_detail.svg'}")

### Heatmap: Evaluierungsmetriken aller Strategien

Diese Übersichtsheatmap zeigt alle wesentlichen Metriken für die wichtigsten Strategien auf einen Blick.

In [ ]:
# Focus on key strategies
key_full = [
    'Ø Alle Judges (roh)', 'Median (roh)', 'Getrimmtes Mittel (roh)',
    'Ø Bias-korrigiert', 'Ø Z-Score-norm.', 'Ø Kalibriert (LOO)',
] + [f'Nur {m} (roh)' for m in models] + [f'{m} (kalibriert LOO)' for m in models]

key_results = df_eval[df_eval['name'].isin(key_full)].copy()
key_results = key_results.set_index('name')

if len(key_results) > 0:
    fig, ax = plt.subplots(figsize=(10, 10))
    heatmap_cols = ['pearson', 'spearman', 'mae', 'agr5', 'agr7']
    display_cols = ['Pearson r', 'Spearman ρ', 'MAE', 'Agr(τ=5)', 'Agr(τ=7)']

    data_for_heatmap = key_results[heatmap_cols].copy()
    data_for_heatmap.columns = display_cols

    # Normalize for coloring (higher is better for all except MAE)
    norm_data = data_for_heatmap.copy()
    norm_data['MAE'] = -norm_data['MAE']  # Invert so higher = better

    sns.heatmap(data_for_heatmap, annot=True, fmt='.3f', cmap='RdYlGn', ax=ax,
                center=0, linewidths=0.5)
    ax.set_title('Evaluierungsmetriken pro Strategie', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'strategy_heatmap.svg', bbox_inches='tight')
    plt.close()
    print(f"Gespeichert: {OUTPUT_DIR / 'strategy_heatmap.svg'}")

---
## Gesamtergebnis

Die folgende Ausgabe fasst die zentralen Befunde für RQ-1 zusammen und bildet die Grundlage für die Diskussion in der Thesis.

In [ ]:
print("\n\n" + "=" * 70)
print("ZUSAMMENFASSUNG DER ERWEITERTEN ANALYSE")
print("=" * 70)

print(f"""
1. AGREEMENT (Ø aller Judges vs. Studie, Total Score):
   - Exakt (τ=0):  {np.sum(np.abs(df_comp['judge_total'].values - study_totals) <= 0)/12:.3f}
   - τ=3:          {np.sum(np.abs(df_comp['judge_total'].values - study_totals) <= 3)/12:.3f}
   - τ=5:          {np.sum(np.abs(df_comp['judge_total'].values - study_totals) <= 5)/12:.3f}
   - τ=7:          {np.sum(np.abs(df_comp['judge_total'].values - study_totals) <= 7)/12:.3f}

2. BESTE STRATEGIE nach MAE:
   {best_row['name']}
   MAE = {best_row['mae']:.2f}, Spearman ρ = {best_row['spearman']:.3f}
   Agreement(τ=5) = {best_row['agr5']:.3f}

3. TOP-3 NACH SPEARMAN ρ:""")

top3_s = df_eval.nlargest(3, 'spearman')
for _, row in top3_s.iterrows():
    print(f"   - {row['name']}: ρ = {row['spearman']:.3f}, MAE = {row['mae']:.2f}")

print(f"""
4. BIAS-KORREKTUR EFFEKT:
   MAE (roh):             {df_eval[df_eval['name']=='Ø Alle Judges (roh)']['mae'].values[0]:.2f}
   MAE (bias-korrigiert): {df_eval[df_eval['name']=='Ø Bias-korrigiert']['mae'].values[0]:.2f}
   MAE (Z-Score-norm.):   {df_eval[df_eval['name']=='Ø Z-Score-norm.']['mae'].values[0]:.2f}
   MAE (kalibriert LOO):  {df_eval[df_eval['name']=='Ø Kalibriert (LOO)']['mae'].values[0]:.2f}

5. KERNBEFUND:
   Die Bias-Korrektur/Kalibrierung kann den absoluten Fehler reduzieren,
   aber die RANGFOLGE (Spearman ρ) bleibt problematisch — die Judges
   bewerten die relative Qualität der Visualisierungen nicht konsistent
   im Vergleich zu menschlichen Bewertern.
""")